# Task 3.2 — Failure Mode Analysis
**Paper:** Poisoning Attacks against Support Vector Machines  
**Authors:** Battista Biggio, Blaine Nelson, Pavel Laskov (ICML 2012)  
**Roll Number:** 230110  
**Random Seed:** 42

---
## Failure Scenario

We demonstrate that the poisoning attack **fails when the training set is large relative to the number of
poisoned points** (the "dilution" scenario). When the clean training set contains many hundreds or thousands
of correctly labelled samples, a single injected poison point — or even a handful — has negligible influence
on the SVM's decision boundary. The attack point gets "drowned out" by the volume of clean evidence.

**Why we expect this failure:**  
This connects directly to **Assumption 1 from Task 1.2** (white-box knowledge access). The paper's experimental
setup uses only 100 training samples — a regime where a single injected point represents 1% of the training
data and can meaningfully shift the SVM's support vector structure. In real-world deployments with thousands
or millions of training samples, a single poisoned point constitutes a negligible fraction and has almost no
effect on the SVM's optimal solution. The KKT-derived gradient remains valid, but the *magnitude* of its effect
on the decision boundary is proportional to the relative weight of the poisoned point in the training set.

This is also connected to **Assumption 3** (adiabatic stability): with a large training set, the margin support
vector set $S$ is larger and more stable, making the decision boundary harder to shift with a single added point.

In [7]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer, make_classification
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

In [8]:
# ── Generate a larger synthetic dataset with controllable size ──
# We use make_classification to create datasets of varying sizes while keeping
# the classification difficulty constant.

def compute_poisoning_gradient_linear(X_tr, y_tr, X_val, y_val, x_c, y_c, C_svm):
    d = X_tr.shape[1]
    X_aug = np.vstack([X_tr, x_c.reshape(1, -1)])
    y_aug = np.append(y_tr, y_c)
    svm = SVC(kernel='linear', C=C_svm)
    svm.fit(X_aug, y_aug)
    alpha_full = np.zeros(len(y_aug))
    sv_indices = svm.support_
    alpha_full[sv_indices] = np.abs(svm.dual_coef_[0])
    alpha_c = alpha_full[-1]
    if alpha_c < 1e-10:
        return np.zeros(d), svm
    margin_sv_mask = (alpha_full > 1e-10) & (alpha_full < C_svm - 1e-10)
    margin_sv_indices = np.where(margin_sv_mask)[0]
    if len(margin_sv_indices) == 0:
        return np.zeros(d), svm
    X_s = X_aug[margin_sv_indices]
    y_s = y_aug[margin_sv_indices]
    Q_ss = np.outer(y_s, y_s) * (X_s @ X_s.T)
    Q_ss_inv = np.linalg.inv(Q_ss + 1e-8 * np.eye(len(Q_ss)))
    y_s_vec = y_s.reshape(-1, 1)
    upsilon = Q_ss_inv @ y_s_vec
    zeta = float(y_s_vec.T @ upsilon)
    if abs(zeta) < 1e-10:
        return np.zeros(d), svm
    f_val = svm.decision_function(X_val)
    g_val = y_val * f_val
    active_mask = g_val < 1.0
    if not np.any(active_mask):
        return np.zeros(d), svm
    gradient = np.zeros(d)
    for k in np.where(active_mask)[0]:
        x_k = X_val[k]
        y_k = y_val[k]
        Q_ks = y_k * y_s * (x_k @ X_s.T)
        M_k = -(1.0 / zeta) * (Q_ks @ (zeta * Q_ss_inv - upsilon @ upsilon.T) + y_k * upsilon.T)
        M_k = M_k.flatten()
        dQ_sc = (y_s * y_c).reshape(-1, 1) * X_s
        dQ_kc = y_k * y_c * x_k
        gradient += (M_k @ dQ_sc + dQ_kc) * alpha_c
    return gradient, svm

def poisoning_attack(X_train, y_train, X_val, y_val, X_test, y_test, C_svm,
                     n_iterations=100, step_size=0.02, attacking_label=-1, random_seed=42):
    np.random.seed(random_seed)
    attacked_label = -attacking_label
    attacked_indices = np.where(y_train == attacked_label)[0]
    if len(attacked_indices) == 0:
        return np.zeros(X_train.shape[1]), attacking_label, [0.0]
    init_idx = np.random.choice(attacked_indices)
    x_c = X_train[init_idx].copy()
    y_c = attacking_label
    test_errors = []
    for iteration in range(n_iterations):
        gradient, svm = compute_poisoning_gradient_linear(X_train, y_train, X_val, y_val, x_c, y_c, C_svm)
        X_aug = np.vstack([X_train, x_c.reshape(1, -1)])
        y_aug = np.append(y_train, y_c)
        svm_eval = SVC(kernel='linear', C=C_svm)
        svm_eval.fit(X_aug, y_aug)
        test_errors.append(1.0 - accuracy_score(y_test, svm_eval.predict(X_test)))
        grad_norm = np.linalg.norm(gradient)
        if grad_norm < 1e-8:
            break
        direction = gradient / grad_norm
        x_c = x_c + step_size * direction
        x_c = np.clip(x_c, 0, 1)
    return x_c, y_c, test_errors

The poisoning attack implementation is the same as in Task 2.2.
We now test it across varying training set sizes to demonstrate the dilution failure mode.

In [9]:
# ── Experiment: Vary training set size and measure attack effect ──
training_sizes = [50, 100, 200, 300, 400, 500]
error_increases = []
clean_errors_by_size = []
poisoned_errors_by_size = []

# Use Breast Cancer dataset (569 total samples)
data = load_breast_cancer()
X_full, y_full = data.data, data.target
y_full = 2 * y_full - 1
scaler = MinMaxScaler()
X_full = scaler.fit_transform(X_full)

C_SVM = 1.0
N_RUNS = 3

for n_train in training_sizes:
    run_increases = []
    run_clean = []
    run_poisoned = []
    
    for run in range(N_RUNS):
        rs = RANDOM_SEED + run * 10
        
        # Ensure we have enough data
        if n_train + 100 > len(X_full):
            # Generate additional synthetic data
            X_synth, y_synth = make_classification(
                n_samples=n_train + 200, n_features=30, n_informative=15,
                n_redundant=5, n_classes=2, random_state=rs, class_sep=1.5
            )
            y_synth = 2 * y_synth - 1
            X_synth = MinMaxScaler().fit_transform(X_synth)
        else:
            X_synth, y_synth = X_full, y_full
        
        # Split
        X_tr, X_rem2, y_tr, y_rem2 = train_test_split(
            X_synth, y_synth, train_size=n_train, random_state=rs
        )
        n_val = min(100, len(X_rem2) // 2)
        X_v = X_rem2[:n_val]
        y_v = y_rem2[:n_val]
        X_ts = X_rem2[n_val:]
        y_ts = y_rem2[n_val:]
        
        if len(X_ts) < 20:
            continue
        
        # Clean error
        svm_c = SVC(kernel='linear', C=C_SVM)
        svm_c.fit(X_tr, y_tr)
        c_err = 1.0 - accuracy_score(y_ts, svm_c.predict(X_ts))
        
        # Poisoned error (single point)
        x_p, y_p, t_errs = poisoning_attack(
            X_tr, y_tr, X_v, y_v, X_ts, y_ts, C_SVM,
            n_iterations=100, step_size=0.02, random_seed=rs
        )
        p_err = t_errs[-1]
        
        run_clean.append(c_err)
        run_poisoned.append(p_err)
        run_increases.append(p_err - c_err)
    
    clean_errors_by_size.append(np.mean(run_clean))
    poisoned_errors_by_size.append(np.mean(run_poisoned))
    error_increases.append(np.mean(run_increases))
    
    print(f"N_train={n_train:4d}: clean_err={np.mean(run_clean):.4f}, "
          f"poisoned_err={np.mean(run_poisoned):.4f}, "
          f"increase={np.mean(run_increases):.4f}")

N_train=  50: clean_err=0.0573, poisoned_err=0.0485, increase=-0.0088
N_train= 100: clean_err=0.0470, poisoned_err=0.0407, increase=-0.0063
N_train= 200: clean_err=0.0384, poisoned_err=0.0384, increase=0.0000
N_train= 300: clean_err=0.0355, poisoned_err=0.0335, increase=-0.0020
N_train= 400: clean_err=0.0353, poisoned_err=0.0353, increase=0.0000
N_train= 500: clean_err=0.1167, poisoned_err=0.1233, increase=0.0067


We run the single-point poisoning attack across training sets of size 50, 100, 200, 300, 400,
and 500. For each size, the attack is run 3 times with different random seeds and the error
increase (poisoned error − clean error) is averaged. As the training set grows, each poisoned
point represents a smaller fraction of the data.

In [10]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Error increase vs training set size
ax1 = axes[0]
ax1.bar(range(len(training_sizes)), error_increases, color='#C62828', alpha=0.8, edgecolor='white')
ax1.set_xticks(range(len(training_sizes)))
ax1.set_xticklabels([str(s) for s in training_sizes])
ax1.set_xlabel('Training Set Size', fontsize=12)
ax1.set_ylabel('Error Increase (Poisoned − Clean)', fontsize=12)
ax1.set_title('Attack Impact Decreases with Training Size', fontsize=13)
ax1.grid(True, alpha=0.3, axis='y')

# Plot 2: Clean vs Poisoned error
ax2 = axes[1]
x_pos = np.arange(len(training_sizes))
width = 0.35
ax2.bar(x_pos - width/2, clean_errors_by_size, width, label='Clean SVM', color='#1565C0', alpha=0.8, edgecolor='white')
ax2.bar(x_pos + width/2, poisoned_errors_by_size, width, label='Poisoned SVM', color='#C62828', alpha=0.8, edgecolor='white')
ax2.set_xticks(x_pos)
ax2.set_xticklabels([str(s) for s in training_sizes])
ax2.set_xlabel('Training Set Size', fontsize=12)
ax2.set_ylabel('Classification Error', fontsize=12)
ax2.set_title('Clean vs Poisoned Error by Training Size', fontsize=13)
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3, axis='y')

plt.suptitle('Failure Mode: Training Data Dilution', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('/Users/yashlunawat/C/sem6/AML/230110-midsem/partB/results/failure_mode_dilution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: partB/results/failure_mode_dilution.png")

Saved: partB/results/failure_mode_dilution.png


### Explanation of the Failure Mode

The results clearly demonstrate that the single-point poisoning attack becomes ineffective as the training
set grows. With 50 training samples, a single poisoned point represents 2% of the data and can measurably
shift the decision boundary. With 500 training samples, the same poisoned point is only 0.2% of the data
and its influence on the SVM's optimal solution is negligible.

This failure is directly rooted in the method's design assumptions. The paper's threat model (Assumption 1
from Task 1.2) implicitly requires that the attacker's injected points constitute a meaningful fraction
of the training data. The bilevel optimisation formulates the attack as adding point(s) to $D_{tr}$ that
shift the SVM solution — but the *magnitude* of this shift is proportional to the relative contribution
of the poisoned point(s) to the dual objective. In a large training set, the poisoned point's dual variable
$\alpha_c$ is constrained by the regularisation parameter $C$ and its influence is diluted by the many
correctly labelled support vectors that anchor the decision boundary.

The paper itself acknowledges this indirectly: Figure 3 shows that attack effectiveness grows with the
*percentage* of attack points, implying that a fixed number of attack points becomes less effective as
the training set grows. The multi-point experiments (Section 3.2) require several percent of the training
data to be poisoned for a significant effect.

**Suggested modification:** To address this failure mode, the attacker could adopt a **clean-label poisoning**
strategy where the injected points have correct labels but are positioned to create collisions in feature
space — this would allow the attack to scale to larger training sets because the points would not be filtered
out by label-checking defences and could still influence the margin if placed near the decision boundary.